# Bank Client Risk Classifier — Full ML Pipeline 🏦
**Портфолио-проект (classification, imbalanced data):** полный пайплайн оценки риска клиента,
от данных до выбора порога по «стоимости ошибок в деньгах». Данные синтетические, проект учебный.

## Этапы
1. Данные → 2. EDA → 3. Preprocessing → 4. Stratified split → 5. Baseline + 3 модели
→ 6. Метрики для несбалансированных классов (PR, ROC-AUC) → 7. Подбор порога по цене ошибки
→ 8. Error analysis → 9. Выводы

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (classification_report, confusion_matrix, ConfusionMatrixDisplay,
                             roc_auc_score, roc_curve, precision_recall_curve, f1_score)

In [ ]:
# ===== 1. ДАННЫЕ: 3000 клиентов, ~12% рискованных =====
rng = np.random.default_rng(77)
n = 3000
employment = rng.choice(["найм", "ИП", "безработный"], n, p=[0.7, 0.2, 0.1])
emp_risk = pd.Series(employment).map({"найм": 0.0, "ИП": 0.4, "безработный": 1.3}).values

df = pd.DataFrame({
    "доход_тыс":        rng.normal(450, 170, n).clip(80).round(0),
    "долг_к_доходу":    rng.uniform(0, 0.95, n).round(2),
    "просрочки_за_год": rng.poisson(0.5, n),
    "стаж_клиента_лет": rng.uniform(0, 18, n).round(1),
    "возраст":          rng.integers(21, 70, n),
    "занятость":        employment,
})
logit = (3.0*df["долг_к_доходу"] + 1.0*df["просрочки_за_год"] + emp_risk
         - 0.003*df["доход_тыс"] - 0.07*df["стаж_клиента_лет"] - 2.0)
df["риск"] = (rng.uniform(0, 1, n) < 1/(1+np.exp(-logit))).astype(int)

# реализм: 3% пропусков в доходе
df.loc[rng.choice(n, int(n*0.03), replace=False), "доход_тыс"] = np.nan
df.to_csv("data_bank_clients.csv", index=False)
print("Баланс классов:"); print(df["риск"].value_counts(normalize=True).round(3))
df.head()

In [ ]:
# ===== 2. EDA =====
print(df.describe().round(1))
print("\nПропуски:\n", df.isna().sum())

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
df.boxplot(column="долг_к_доходу", by="риск", ax=axes[0])
df.boxplot(column="доход_тыс", by="риск", ax=axes[1])
risk_by_emp = df.groupby("занятость")["риск"].mean()
axes[2].bar(risk_by_emp.index, risk_by_emp.values); axes[2].set_title("Доля риска по занятости")
plt.tight_layout(); plt.show()

### Наблюдения EDA
- У рискованных клиентов заметно выше долг/доход и ниже доход — признаки информативны.
- «Безработный» — в разы выше доля риска: категориальный признак обязателен в модели.
- Классы 88/12 — несбалансированность. Accuracy будет лгать; работаем с recall/PR/ROC-AUC
  и обязательно stratify при split.

In [ ]:
# ===== 3-4. PREPROCESSING + STRATIFIED SPLIT =====
X = df.drop(columns="риск"); y = df["риск"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

num_cols = ["доход_тыс", "долг_к_доходу", "просрочки_за_год", "стаж_клиента_лет", "возраст"]
cat_cols = ["занятость"]
prep = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("sc", StandardScaler())]), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])

In [ ]:
# ===== 5-6. BASELINE + 3 МОДЕЛИ =====
print("Baseline 'все надёжные': accuracy =", (y_te == 0).mean().round(3),
      "| recall риска = 0.0  <- бесполезна, хотя accuracy 'красивая'\n")

models = {
    "LogReg balanced": LogisticRegression(class_weight="balanced", max_iter=1000),
    "RandomForest":    RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=0),
    "GradBoosting":    GradientBoostingClassifier(random_state=0),
}
fitted, rows = {}, []
for name, m in models.items():
    pipe = Pipeline([("prep", prep), ("model", m)]).fit(X_tr, y_tr)
    proba = pipe.predict_proba(X_te)[:, 1]
    pred = (proba >= 0.5).astype(int)
    fitted[name] = (pipe, proba)
    rows.append({"model": name,
                 "ROC-AUC": roc_auc_score(y_te, proba).round(3),
                 "F1(риск)": f1_score(y_te, pred).round(3),
                 "recall(риск)": classification_report(y_te, pred, output_dict=True)["1"]["recall"].__round__(3)})
print(pd.DataFrame(rows).set_index("model"))

In [ ]:
# ROC и Precision-Recall кривые
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for name, (pipe, proba) in fitted.items():
    fpr, tpr, _ = roc_curve(y_te, proba)
    axes[0].plot(fpr, tpr, label=f"{name} (AUC={roc_auc_score(y_te, proba):.2f})")
    pr, rc, _ = precision_recall_curve(y_te, proba)
    axes[1].plot(rc, pr, label=name)
axes[0].plot([0,1],[0,1],'k--'); axes[0].set_title('ROC'); axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR'); axes[0].legend()
axes[1].set_title('Precision-Recall'); axes[1].set_xlabel('recall'); axes[1].set_ylabel('precision'); axes[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
# ===== 7. ПОДБОР ПОРОГА ПО ДЕНЬГАМ =====
# Цена ошибок: пропустить рискового (FN) = -2000$, зря проверить хорошего (FP) = -100$
best_name = max(fitted, key=lambda k: roc_auc_score(y_te, fitted[k][1]))
pipe, proba = fitted[best_name]
print("Лучшая по AUC:", best_name)

thresholds = np.linspace(0.05, 0.95, 50)
costs = []
for t in thresholds:
    pred = (proba >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_te, pred).ravel()
    costs.append(fn * 2000 + fp * 100)
best_t = thresholds[int(np.argmin(costs))]

plt.plot(thresholds, costs); plt.axvline(best_t, c='r', ls='--', label=f'опт. порог {best_t:.2f}')
plt.axvline(0.5, c='gray', ls=':', label='стандартный 0.5')
plt.xlabel('порог'); plt.ylabel('убыток, $'); plt.legend(); plt.grid(True)
plt.title('Дешевле ловить риск агрессивнее, чем при пороге 0.5'); plt.show()

pred_best = (proba >= best_t).astype(int)
print(f"Порог {best_t:.2f}: убыток {min(costs)}$  vs  порог 0.50: убыток {costs[np.argmin(np.abs(thresholds-0.5))]}$")
ConfusionMatrixDisplay(confusion_matrix(y_te, pred_best), display_labels=["надёжный","риск"]).plot(cmap='Oranges')
plt.title(f'{best_name}, порог {best_t:.2f}'); plt.show()

In [ ]:
# ===== 8. ERROR ANALYSIS: кого модель всё ещё пропускает? =====
miss_mask = (y_te.values == 1) & (pred_best == 0)
missed = X_te[miss_mask].copy(); missed["P(риск)"] = proba[miss_mask].round(2)
print(f"Пропущено рисковых: {miss_mask.sum()} из {int(y_te.sum())}")
print(missed.describe().round(2).loc[["mean", "50%"]])
print("\nСравни со средним по всем рисковым:")
print(X_te[y_te.values == 1].describe().round(2).loc[["mean"]])

### Разбор ошибок
- Пропущенные рисковые — «тихие»: долг и просрочки у них ниже, чем у типичного рискового клиента.
  Их сигналы просто отсутствуют в наших признаках → нужны новые источники (кредитная история
  из бюро, транзакционное поведение).
- Главный практический результат: **подбор порога по стоимости ошибок сократил убыток банка**
  по сравнению с бездумным порогом 0.5 — это и есть ML на службе бизнес-метрики.

## ===== 9. ВЫВОДЫ =====
1. На несбалансированных данных пайплайн строится вокруг recall/PR/AUC, stratify и class_weight.
2. Порог классификации — бизнес-решение, а не константа 0.5: мы выбрали его минимизацией убытка в $.
3. Error analysis указал направление улучшения данных, а не бесконечный тюнинг модели.

## Идеи улучшения до production-level
- SMOTE/undersampling и сравнение с class_weight; калибровка вероятностей (CalibratedClassifierCV)
- Optuna для гиперпараметров GBM; SHAP для объяснения отказов клиентам (регуляторика!)
- Monitoring: дрейф признаков, переобучение модели по расписанию
- REST-сервис скоринга + журналирование решений

---
## Задания для практики 💪
1. Поменяй стоимость ошибок (FN=500$, FP=200$) и найди новый оптимальный порог. Куда он сдвинулся и почему?
2. Добавь SMOTE (pip install imbalanced-learn) вместо class_weight и сравни recall/precision.
3. Выведи feature_importances_ RandomForest — совпадают ли главные признаки с выводами EDA?
4. Сделай калибровочную кривую (sklearn.calibration.calibration_curve) для лучшей модели: можно ли верить её вероятностям?

## 5 проверочных вопросов ❓
1. Почему baseline «все надёжные» с accuracy 0.88 бесполезен и какие метрики мы выбрали вместо accuracy?
2. Зачем stratify=y при split на несбалансированных данных?
3. Что показывает ROC-кривая, а что Precision-Recall, и какая информативнее при сильном дисбалансе?
4. Почему порог 0.5 — не закон, и как мы выбирали порог в этом проекте?
5. Кого модель пропускает даже при низком пороге и каков правильный следующий шаг (подсказка: не тюнинг модели)?

## Чек-лист перед выкладкой на GitHub ✅
- [ ] Прогнал ноутбук сверху вниз без ошибок (Restart & Run All)
- [ ] Переписал выводы своими словами
- [ ] Проверил src/-скрипты по цепочке generate_data → train → predict
- [ ] Заполнил description и topics из README